<a href="https://colab.research.google.com/github/fengli949/Coursera_Introduction_to_data_science/blob/master/hg19_to_hg38_coordinates_by_Python_Liftover_package.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
ls

sample_data/


In [ ]:
!pip install pyliftover pandas
import pandas as pd
from pyliftover import LiftOver

In [15]:
# convert chromosome coordinates from Hg19 to Hg 38 using pyliftover
# 1. Initialize the converter (auto-downloads hg19->hg38 chain on first run)
lo = LiftOver('hg19', 'hg38')

# 2. Load your original hg19 table
df = pd.read_csv('snp_sbs_xrcc4_clinical_merged.txt', sep='\t')  # Adjust filename/separator as needed

def lift_coordinate(row):
    # Handle NaN in position
    if pd.isna(row['Physical Position']):
        return pd.Series([None, None])

    # Ensure chromosome format matches 'chr5'
    chrom = str(row['Chromosome']).strip()
    if not chrom.startswith('chr'):
        chrom = 'chr' + chrom

    try:
        pos = int(float(row['Physical Position']))  # handles values stored as "12345.0"
    except (ValueError, TypeError):
        return pd.Series([None, None])

    # Perform the lift (convert_coordinate expects 0-based position)
    new_coord = lo.convert_coordinate(chrom, pos - 1)

    if new_coord:
        new_chrom, new_pos = new_coord[0][0], new_coord[0][1]
        return pd.Series([new_chrom, new_pos + 1])  # back to 1-based
    else:
        return pd.Series([None, None])

# 3. Apply the function to generate new columns
df[['chr_hg38', 'position_hg38']] = df.apply(lift_coordinate, axis=1)

# 4. Save your updated table
df.to_csv('my_hg38_converted_table.csv', index=False)
print("Conversion complete!")

Conversion complete!
